# Phase 1 Final A2d Covariance/Tangent Runner

This notebook is an orchestration and audit surface only. Durable scientific logic lives in `src/` and is exercised through `python -m src.cli phase1_final_a2d_runner`.

Scientific integrity boundary:

- This runner is intended to address the A2d missing-output blocker recorded by the final feature-matrix comparator runner.
- It must not approximate A2d from bandpower features.
- It uses the reviewed final feature matrix row index only as row/provenance contract, then extracts covariance matrices from final EDF payloads.
- Log-Euclidean reference, tangent projection, normalization and classifier fitting must use training subjects only for each LOSO fold.
- Claims remain closed. Metrics are implementation diagnostics only until the full comparator/control/calibration/influence/reporting package passes.

In [ ]:
# Cell 1 - Bootstrap repo, optionally install signal extras, and run unit tests.

from __future__ import annotations

import getpass
import hashlib
import importlib.util
import json
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
INSTALL_SIGNAL_EXTRAS = False
RUN_UNITTESTS = True

def run(cmd, cwd=None, check=True, env=None):
    printable = ['<redacted>' if 'Authorization:' in str(part) else str(part) for part in cmd]
    print('$', ' '.join(printable))
    return subprocess.run(cmd, cwd=cwd, check=check, text=True, env=env)

if not REPO_DIR.exists():
    token = getpass.getpass('Enter GitHub token for private repo, or leave blank for public clone: ')
    if token:
        header = 'Authorization: Basic ' + __import__('base64').b64encode(f'x-access-token:{token}'.encode()).decode()
        run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
    else:
        run(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

if INSTALL_SIGNAL_EXTRAS:
    env = os.environ.copy()
    env['INSTALL_SIGNAL_EXTRAS'] = '1'
    run(['bash', 'bootstrap/install_runtime.sh'], cwd=REPO_DIR, env=env)

print('Repo:', REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)

if RUN_UNITTESTS:
    unit = subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, text=True)
    if unit.returncode != 0:
        raise subprocess.CalledProcessError(unit.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])

In [ ]:
# Cell 2 - Select reviewed sources and keep launch disabled by default.

EXPECTED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'

PREREG_BUNDLE = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z/prereg_bundle.json'
DATASET_ROOT = DRIVE_ROOT / 'data/ds004752'
FINAL_SPLIT_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_split_manifest'
FINAL_FEATURE_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_feature_manifest'
FINAL_LEAKAGE_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_leakage_audit'
FEATURE_MATRIX_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_feature_matrix'
FEATURE_MATRIX_COMPARATOR_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_comparator_runner'
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_a2d_runner'
PLAN_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_a2d_runner_plan'

RUN_FINAL_A2D_RUNNER = False
REQUIRED_ACK = 'I understand this final A2d runner is claim-closed and must not be interpreted as efficacy evidence'
MANUAL_LAUNCH_ACK = ''

def latest_run(root: Path) -> Path:
    latest = root / 'latest.txt'
    if latest.exists():
        return Path(latest.read_text(encoding='utf-8').strip())
    runs = sorted(path for path in root.iterdir() if path.is_dir())
    if not runs:
        raise FileNotFoundError(f'No run dirs found under {root}')
    return runs[-1]

FINAL_SPLIT_RUN = latest_run(FINAL_SPLIT_ROOT)
FINAL_FEATURE_RUN = latest_run(FINAL_FEATURE_ROOT)
FINAL_LEAKAGE_RUN = latest_run(FINAL_LEAKAGE_ROOT)
FEATURE_MATRIX_RUN = latest_run(FEATURE_MATRIX_ROOT)
FEATURE_MATRIX_COMPARATOR_RUN = latest_run(FEATURE_MATRIX_COMPARATOR_ROOT)

print('Prereg bundle:', PREREG_BUNDLE)
print('Dataset root:', DATASET_ROOT)
print('Final split run:', FINAL_SPLIT_RUN)
print('Final feature run:', FINAL_FEATURE_RUN)
print('Final leakage run:', FINAL_LEAKAGE_RUN)
print('Feature matrix run:', FEATURE_MATRIX_RUN)
print('Feature-matrix comparator run:', FEATURE_MATRIX_COMPARATOR_RUN)
print('Output root:', OUTPUT_ROOT)
print('Run flag:', RUN_FINAL_A2D_RUNNER)

In [ ]:
# Cell 3 - Validate source boundaries before any launch.

def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_sha256 = sha256(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
assert bundle.get('status') == 'locked'
assert actual_prereg_hash == EXPECTED_PREREG_IDENTITY_HASH

split_summary = read_json(FINAL_SPLIT_RUN / 'phase1_final_split_manifest_summary.json')
feature_summary = read_json(FINAL_FEATURE_RUN / 'phase1_final_feature_manifest_summary.json')
leakage_summary = read_json(FINAL_LEAKAGE_RUN / 'phase1_final_leakage_audit_summary.json')
leakage_audit = read_json(FINAL_LEAKAGE_RUN / 'final_leakage_audit.json')
matrix_summary = read_json(FEATURE_MATRIX_RUN / 'phase1_final_feature_matrix_summary.json')
matrix_schema = read_json(FEATURE_MATRIX_RUN / 'phase1_final_feature_matrix_schema.json')
matrix_validation = read_json(FEATURE_MATRIX_RUN / 'phase1_final_feature_matrix_validation.json')
previous_summary = read_json(FEATURE_MATRIX_COMPARATOR_RUN / 'phase1_final_comparator_runner_summary.json')
mne_available = importlib.util.find_spec('mne') is not None

print('Split ready:', split_summary.get('split_manifest_ready'))
print('Feature ready:', feature_summary.get('feature_manifest_ready'))
print('Leakage ready:', leakage_summary.get('leakage_audit_ready'))
print('Feature matrix ready:', matrix_summary.get('feature_matrix_ready'))
print('Feature matrix rows/features:', matrix_summary.get('n_rows'), matrix_summary.get('n_features'))
print('Previous runner blocked comparators:', previous_summary.get('blocked_comparators'))
print('mne available:', mne_available)

assert split_summary.get('status') == 'phase1_final_split_manifest_recorded'
assert split_summary.get('split_manifest_ready') is True
assert feature_summary.get('status') == 'phase1_final_feature_manifest_recorded'
assert feature_summary.get('feature_manifest_ready') is True
assert leakage_summary.get('status') == 'phase1_final_leakage_audit_recorded'
assert leakage_summary.get('leakage_audit_ready') is True
assert leakage_audit.get('outer_test_subject_used_in_any_fit') is False
assert leakage_audit.get('test_time_privileged_or_teacher_outputs_allowed') is False
assert matrix_summary.get('status') == 'phase1_final_feature_matrix_materialized'
assert matrix_summary.get('feature_matrix_ready') is True
assert matrix_validation.get('status') == 'phase1_final_feature_matrix_validation_passed'
assert matrix_summary.get('contains_model_outputs') is False
assert matrix_summary.get('contains_logits') is False
assert matrix_summary.get('contains_metrics') is False
assert matrix_schema.get('contains_model_outputs') is False
assert matrix_schema.get('contains_logits') is False
assert matrix_schema.get('contains_metrics') is False
assert 'A2d_riemannian' in previous_summary.get('blocked_comparators', []), 'Previous runner must document A2d blocker before this step.'
if RUN_FINAL_A2D_RUNNER and not mne_available:
    raise RuntimeError('Final A2d EDF covariance extraction requires mne/signal extras. Set INSTALL_SIGNAL_EXTRAS=True in Cell 1, rerun bootstrap/tests, then relaunch intentionally.')

In [ ]:
# Cell 4 - Record a launch plan with explicit non-claim boundary.

created_utc = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
PLAN_DIR = PLAN_ROOT / created_utc
PLAN_DIR.mkdir(parents=True, exist_ok=False)

help_result = subprocess.run(
    ['python', '-m', 'src.cli', 'phase1_final_a2d_runner', '--help'],
    cwd=REPO_DIR,
    text=True,
    capture_output=True,
)
runner_available = help_result.returncode == 0 and '--feature-matrix-run' in help_result.stdout

cmd = [
    'python', '-m', 'src.cli', 'phase1_final_a2d_runner',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--final-split-run', str(FINAL_SPLIT_RUN),
    '--final-feature-run', str(FINAL_FEATURE_RUN),
    '--final-leakage-run', str(FINAL_LEAKAGE_RUN),
    '--feature-matrix-run', str(FEATURE_MATRIX_RUN),
    '--feature-matrix-comparator-run', str(FEATURE_MATRIX_COMPARATOR_RUN),
    '--dataset-root', str(DATASET_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--runner-config', 'configs/phase1/final_a2d_runner.json',
]

plan = {
    'status': 'phase1_final_a2d_runner_plan_recorded',
    'created_utc': created_utc,
    'runner_available': runner_available,
    'run_flag': RUN_FINAL_A2D_RUNNER,
    'required_ack': REQUIRED_ACK,
    'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    'mne_available': mne_available,
    'prereg_identity_hash': actual_prereg_hash,
    'raw_prereg_file_sha256': raw_prereg_sha256,
    'final_split_run': str(FINAL_SPLIT_RUN),
    'final_feature_run': str(FINAL_FEATURE_RUN),
    'final_leakage_run': str(FINAL_LEAKAGE_RUN),
    'feature_matrix_run': str(FEATURE_MATRIX_RUN),
    'feature_matrix_comparator_run': str(FEATURE_MATRIX_COMPARATOR_RUN),
    'dataset_root': str(DATASET_ROOT),
    'planned_command': cmd,
    'scientific_integrity_boundary': {
        'claim_ready': False,
        'headline_phase1_claim_open': False,
        'uses_bandpower_features_as_a2d_input': False,
        'uses_final_feature_matrix_row_index_as_contract': True,
        'extracts_covariance_from_edf_payloads': True,
        'not_ok_to_claim': [
            'decoder efficacy',
            'A2d final comparator efficacy',
            'A4 privileged-transfer efficacy',
            'full Phase 1 neural comparator performance',
        ],
    },
}
(PLAN_DIR / 'phase1_final_a2d_runner_plan.json').write_text(json.dumps(plan, indent=2) + '\n', encoding='utf-8')
print(json.dumps(plan, indent=2))

In [ ]:
# Cell 5 - Manual hold or explicit launch through CLI.

if not runner_available:
    raise RuntimeError('CLI route phase1_final_a2d_runner is unavailable. Pull the correct commit before continuing.')

if not RUN_FINAL_A2D_RUNNER:
    hold = {
        'status': 'phase1_final_a2d_runner_manual_hold',
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'plan_dir': str(PLAN_DIR),
        'run_flag': RUN_FINAL_A2D_RUNNER,
        'required_ack': REQUIRED_ACK,
        'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
        'mne_available': mne_available,
        'next': 'Review the plan. Rerun only after installing signal extras, setting RUN_FINAL_A2D_RUNNER=True, and using the exact acknowledgement.',
    }
    (PLAN_DIR / 'phase1_final_a2d_runner_manual_hold.json').write_text(json.dumps(hold, indent=2) + '\n', encoding='utf-8')
    print(json.dumps(hold, indent=2))
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not launch final A2d runner without explicit non-claim acknowledgement.')
elif not mne_available:
    raise RuntimeError('Final A2d EDF covariance extraction requires mne/signal extras. Set INSTALL_SIGNAL_EXTRAS=True in Cell 1, rerun bootstrap/tests, then relaunch intentionally.')
else:
    run(cmd, cwd=REPO_DIR)
    print('Final A2d covariance/tangent command completed. Review artifacts before any interpretation.')

In [ ]:
# Cell 6 - Inspect latest A2d output if launched.

latest_output = None
summary = None
if RUN_FINAL_A2D_RUNNER:
    latest_output = latest_run(OUTPUT_ROOT)
    print('Latest final A2d output:', latest_output)
    required = [
        'phase1_final_a2d_runner_inputs.json',
        'phase1_final_a2d_runner_summary.json',
        'phase1_final_a2d_runner_report.md',
        'phase1_final_a2d_runner_source_links.json',
        'phase1_final_a2d_runner_input_validation.json',
        'phase1_final_a2d_covariance_validation.json',
        'a2d_final_covariance_manifest.json',
        'a2d_final_tangent_manifest.json',
        'phase1_final_a2d_completeness_patch.json',
        'phase1_final_a2d_claim_state.json',
        'final_logits/A2d_riemannian_final_logits.json',
        'final_subject_level_metrics/A2d_riemannian_subject_level_metrics.json',
        'runtime_leakage_logs/A2d_riemannian_runtime_leakage_audit.json',
        'comparator_output_manifests/A2d_riemannian_output_manifest.json',
    ]
    for name in required:
        print(name, 'exists =', (latest_output / name).exists())
    summary = read_json(latest_output / 'phase1_final_a2d_runner_summary.json')
    print(json.dumps(summary, indent=2))
else:
    print('Manual hold active. No final A2d output inspected in this pass.')

In [ ]:
# Cell 7 - Assertions for launched run.

if RUN_FINAL_A2D_RUNNER:
    assert summary is not None
    assert summary.get('status') == 'phase1_final_a2d_covariance_tangent_runner_complete_claim_closed', summary.get('blockers')
    assert summary.get('a2d_final_output_present') is True
    assert summary.get('claim_ready') is False
    assert summary.get('headline_phase1_claim_open') is False
    assert summary.get('full_phase1_claim_bearing_run_allowed') is False
    assert summary.get('smoke_artifacts_promoted') is False
    assert summary.get('n_covariance_rows') == summary.get('n_expected_rows')
    leakage = read_json(latest_output / 'runtime_leakage_logs/A2d_riemannian_runtime_leakage_audit.json')
    assert leakage.get('outer_test_subject_used_for_any_fit') is False
    assert leakage.get('test_time_privileged_or_teacher_outputs_allowed') is False
    logits = read_json(latest_output / 'final_logits/A2d_riemannian_final_logits.json')
    assert logits.get('contains_covariance_values') is False
    assert logits.get('contains_tangent_features') is False
    assert 'covariance' not in logits.get('rows', [{}])[0]
    claim_state = read_json(latest_output / 'phase1_final_a2d_claim_state.json')
    assert claim_state.get('claim_ready') is False
    assert 'controls_calibration_influence_reporting_missing' in claim_state.get('blockers', [])
else:
    print('Assertions skipped because run flag is False.')

In [ ]:
# Cell 8 - Write non-claim review note only after launched run.

if RUN_FINAL_A2D_RUNNER:
    leakage = read_json(latest_output / 'runtime_leakage_logs/A2d_riemannian_runtime_leakage_audit.json')
    claim_state = read_json(latest_output / 'phase1_final_a2d_claim_state.json')
    patch = read_json(latest_output / 'phase1_final_a2d_completeness_patch.json')
    metrics = read_json(latest_output / 'final_subject_level_metrics/A2d_riemannian_subject_level_metrics.json')
    review = {
        'status': 'phase1_final_a2d_covariance_tangent_review_pass_non_claim',
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'reviewed_run': str(latest_output),
        'scope': 'Final A2d covariance/tangent output package only; claims remain closed.',
        'checks_passed': [
            'required_artifacts_present',
            'covariance_rows_match_final_feature_matrix_row_contract',
            'a2d_uses_edf_covariance_not_bandpower_proxy',
            'runtime_leakage_passed',
            'outer_test_subject_not_used_for_reference_tangent_normalization_classifier_or_calibration_fit',
            'logits_do_not_store_covariance_or_tangent_values',
            'claim_ready_false',
            'headline_claim_closed',
        ],
        'metric_summary_recorded_as_diagnostic_only': {
            'median_balanced_accuracy': metrics.get('median_balanced_accuracy'),
            'mean_ece_10_bins': metrics.get('mean_ece_10_bins'),
            'mean_brier': metrics.get('mean_brier'),
            'claim_ready': metrics.get('claim_ready'),
            'claim_evaluable': metrics.get('claim_evaluable'),
        },
        'runtime_leakage_summary': {
            'outer_test_subject_used_for_any_fit': leakage.get('outer_test_subject_used_for_any_fit'),
            'test_time_privileged_or_teacher_outputs_allowed': leakage.get('test_time_privileged_or_teacher_outputs_allowed'),
        },
        'resolved_blockers_for_downstream_reconciliation': patch.get('resolved_blockers_for_downstream_reconciliation'),
        'metrics_interpretation': {
            'allowed': 'Implementation diagnostics and artifact completeness review only.',
            'not_allowed': 'Do not use A2d metrics as efficacy, superiority, or privileged-transfer evidence.',
        },
        'claim_blockers': claim_state.get('blockers'),
        'not_ok_to_claim': claim_state.get('not_ok_to_claim'),
    }
    note_path = latest_output / 'phase1_final_a2d_runner_review_note.json'
    note_path.write_text(json.dumps(review, indent=2) + '\n', encoding='utf-8')
    print('Review note written:', note_path)
    print(json.dumps(review, indent=2))
else:
    print('Review note not written because run flag is False.')

In [ ]:
# Cell 9 - Closeout.

print('================ Phase 1 Final A2d Covariance/Tangent Closeout ================')
print('Notebook fix marker: phase1_final_a2d_cov_tangent_v1_20260422')
print('Plan source of truth:', PLAN_DIR)
print('Runner available:', runner_available)
print('Run flag:', RUN_FINAL_A2D_RUNNER)
print('mne available:', mne_available)
print('Expected locked prereg identity hash:', EXPECTED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
if RUN_FINAL_A2D_RUNNER:
    print('Latest final A2d output:', latest_output)
    print('A2d output present:', summary.get('a2d_final_output_present'))
    print('Rows:', summary.get('n_covariance_rows'))
    print('Folds:', summary.get('n_folds'))
    print('Claim blockers:', summary.get('claim_blockers'))
    print('CHECK REQUIRED: Review A2d covariance/tangent manifests, logits, subject metrics and runtime leakage logs before downstream reconciliation.')
else:
    print('HELD: Runner appears available, but execution was not launched because manual flag is False.')
    print('NEXT: review the plan, install signal extras if needed, then rerun only with explicit non-claim acknowledgement if appropriate.')
print('\nNOT OK TO CLAIM: This notebook does not prove decoder efficacy, A2d efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')